# Example notebook for UMAP / t-SNE dimensionality reduction plots

The present notebook serves as a guide of how to use the `IDEAL-GENOM-QC` library to draw and analyse UMAP and t-SNE plots of population structure.

The underlying module (`ideal_genom.population.projection`) splits this into composable pieces — `PCAReduction` (LD pruning + PCA), `UMAPReduction`, `TSNEReduction`, and `Plot2D` for metadata-aware plotting — all orchestrated by `DimensionalityReductionPipeline`. We show both a single-parameter run and a parameter-grid sweep (the grid is auto-detected whenever a parameter value is a list instead of a scalar).

Let us import the required libraries.

In [14]:
import sys
import os

import pandas as pd

from pathlib import Path

# add parent directory to path
library_path = os.path.abspath('..')
if library_path not in sys.path:
    sys.path.append(library_path)

library_path = Path(library_path)

from ideal_genom.population.projection import DimensionalityReductionPipeline

In the next cell the path variables associated with the project are set.

As with ancestry QC, this step is typically run on the cleaned output of the sample QC pipeline.

In [27]:
DATA_PATH = library_path / 'ideal_genom' / 'data'

example_data = DATA_PATH / 'example_data'
ouputData    = example_data / 'outputData'

# Use the cleaned output of the variant QC notebook as input
input_path = ouputData / 'variant_qc_results' / 'clean_files'
input_name = '1KG_GRCh38_variant_qc'
output_path = ouputData
high_ld_file = Path('path/to/ld_file') # if not available, set to a non-existent Path() and it will be fetched automatically

In the next cell we define the parameter dictionaries used by the pipeline.

**PCA preparation** (`pca_params`, used for LD pruning + PCA, required before either reduction):

1. `maf`: Parameter `--maf` of **PLINK1.9**.
2. `mind`: Parameter `--mind` of **PLINK1.9**.
3. `geno`: Parameter `--geno` of **PLINK1.9**.
4. `hwe`: Parameter `--hwe` of **PLINK1.9**.
5. `ind_pair`: Parameter `--indep-pairwise` of **PLINK1.9**.
6. `pca`: number of principal components computed; these are the components UMAP/t-SNE will be run on.

**UMAP** (`umap_params`) and **t-SNE** (`tsne_params`): these accept either single scalar values (one run) or lists of values (automatically triggers a grid search over all combinations — see `umap-learn` docs at https://umap-learn.readthedocs.io/en/latest/ and scikit-learn's t-SNE docs for parameter meanings).

**Plotting**:

- `color_hue_file`: optional path to a tab-separated file whose first two columns match the `.fam` file's ID columns, and whose third column is a categorical variable used as plot hue (e.g. the `population_tags` file produced by the ancestry QC notebook).
- `case_control_marker`: if `True`, uses the `.fam` file's case/control phenotype as the plot hue/style instead.

In [28]:
pca_params = {
    'maf': 0.01,
    'mind': 0.2,
    'geno': 0.1,
    'hwe': 5e-8,
    'ind_pair': [20000, 2000, 0.2],
    'pca': 25,
}

# Single-run parameters
umap_params_single = {
    'n_neighbors': 15,
    'min_dist': 0.1,
    'metric': 'euclidean',
    'random_state': 42,
}

tsne_params_single = {
    'perplexity': 30.0,
    'random_state': 42,
}

# Set to a Path, e.g. the `population_tags` file from the ancestry QC notebook, if you want population-colored plots
color_hue_file = None
case_control_marker = False

Initialize the `DimensionalityReductionPipeline`.

In [29]:
pipeline = DimensionalityReductionPipeline(
    input_path =input_path,
    input_name =input_name,
    output_path=output_path,
    build      ='38',
    high_ld_regions_file=high_ld_file,
)

INFO:ideal_genom.population.projection:Pipeline initialized for 1KG_GRCh38_variant_qc
INFO:ideal_genom.population.projection:Results will be saved to /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/dimensionality_reduction_results


First, run a single-parameter pass: PCA preparation, then one UMAP run and one t-SNE run, then the corresponding plots. `execute_dimensionality_reduction_pipeline()` auto-detects that none of the `umap_params`/`tsne_params` values are lists, so it takes the single-run path.

The PCA preparation step shells out to PLINK, which prints a lot of console text; we capture it into `dimred_single_log` to keep the notebook readable — run `dimred_single_log.show()` in a new cell if you need to inspect it.

In [30]:
%%capture dimred_single_log
results_single = pipeline.execute_dimensionality_reduction_pipeline(
    pca_params=pca_params,
    run_umap=True,
    umap_params=umap_params_single,
    run_tsne=True,
    tsne_params=tsne_params_single,
    color_hue_file=color_hue_file,
    case_control_markers=case_control_marker,
    plot_format='svg',
)

INFO:ideal_genom.population.projection:================================================================================
INFO:ideal_genom.population.projection:STARTING FULL DIMENSIONALITY REDUCTION PIPELINE
INFO:ideal_genom.population.projection:================================================================================
INFO:ideal_genom.population.projection:Single parameter set detected - running standard pipeline
INFO:ideal_genom.population.projection:PCA files not found, running PCA preparation
INFO:ideal_genom.population.projection:================================================================================
INFO:ideal_genom.population.projection:STEP 1: PCA Preparation
INFO:ideal_genom.population.projection:================================================================================
INFO:ideal_genom.population.projection:High LD file not provided.
INFO:ideal_genom.population.projection:High LD file will be fetched from the package
INFO:ideal_genom.population.projection

PLINK v2.0.0-a.6.26LM AVX2 Intel (26 Oct 2025)      cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang    GNU General Public License v3
Logging to /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/dimensionality_reduction_results/1KG_GRCh38_variant_qc.log.
Options in effect:
  --bfile /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/variant_qc_results/clean_files/1KG_GRCh38_variant_qc
  --exclude range /home/luis/CGE/ideal-genom-qc/ideal_genom/data/ld_regions_files/high-LD-regions_GRCH38.txt
  --geno 0.1
  --hwe 5e-08
  --indep-pairwise 20000 2000 0.2
  --maf 0.01
  --mind 0.2
  --out /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/dimensionality_reduction_results/1KG_GRCh38_variant_qc
  --threads 30

Start time: Fri Jun 26 12:52:39 2026
63862 MiB RAM detected, ~49835 available; reserving 31931 MiB for main
workspace.
Using up to 30 threads (change this with --threads).
200 samples (94 females, 10

INFO:ideal_genom.core.executor:Command completed successfully
INFO:ideal_genom.core.executor:Executing: plink2 --bfile /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/variant_qc_results/clean_files/1KG_GRCh38_variant_qc --extract /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/dimensionality_reduction_results/1KG_GRCh38_variant_qc.prune.in --threads 30 --maf 0.01 --geno 0.1 --mind 0.2 --hwe 5e-08 --make-bed --out /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/dimensionality_reduction_results/1KG_GRCh38_variant_qc-pruned
INFO:ideal_genom.core.executor:Command completed successfully
INFO:ideal_genom.population.projection:STEP: Performing principal component decomposition
INFO:ideal_genom.core.executor:Executing: plink2 --bfile /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/dimensionality_reduction_results/1KG_GRCh38_variant_qc-pruned --maf 0.01 --out /home/luis/CGE/ideal-genom-qc/ideal_genom/dat

28564/74791 variants removed.
Variant lists written to
/home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/dimensionality_reduction_results/1KG_GRCh38_variant_qc.prune.in
and
/home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/dimensionality_reduction_results/1KG_GRCh38_variant_qc.prune.out
.
End time: Fri Jun 26 12:52:40 2026
PLINK v2.0.0-a.6.26LM AVX2 Intel (26 Oct 2025)      cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang    GNU General Public License v3
Logging to /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/dimensionality_reduction_results/1KG_GRCh38_variant_qc-pruned.log.
Options in effect:
  --bfile /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/variant_qc_results/clean_files/1KG_GRCh38_variant_qc
  --extract /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/dimensionality_reduction_results/1KG_GRCh38_variant_qc.prune.in
  --geno 0.1
  --hwe 5e

INFO:ideal_genom.core.executor:Command completed successfully
INFO:ideal_genom.population.projection:No metadata provided
INFO:ideal_genom.population.projection:No metadata to merge. Plotting 200 samples
INFO:ideal_genom.population.projection:Generating 2D plot: pca_2d_plot
INFO:ideal_genom.population.projection:Plot dimensions: PC1 vs PC2


313233343536373839404142434445464748495051525354555657585960616263646566676869707172737475767778798081828384858687888990919293949596979899done.
Extracting eigenvalues and eigenvectors... done.
--pca: Eigenvectors written to
/home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/dimensionality_reduction_results/1KG_GRCh38_variant_qc-pca.eigenvec
, and eigenvalues written to
/home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/dimensionality_reduction_results/1KG_GRCh38_variant_qc-pca.eigenval
.
End time: Fri Jun 26 12:52:41 2026


INFO:ideal_genom.population.projection:Plot saved to /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/dimensionality_reduction_results/pca_2d_plot
INFO:ideal_genom.population.projection:Preparation pipeline completed successfully.
INFO:ideal_genom.population.projection:PCA preparation completed
INFO:ideal_genom.population.projection:Eigenvector file: /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/dimensionality_reduction_results/1KG_GRCh38_variant_qc-pca.eigenvec
INFO:ideal_genom.population.projection:Eigenvalue file: /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/dimensionality_reduction_results/1KG_GRCh38_variant_qc-pca.eigenval
INFO:ideal_genom.population.projection:================================================================================
INFO:ideal_genom.population.projection:STEP 2: UMAP Reduction
INFO:ideal_genom.population.projection:======================================================================

In [31]:
print("Single-parameter UMAP/t-SNE run completed.")

Single-parameter UMAP/t-SNE run completed.


`results_single` is a summary dict pointing at every file produced (eigenvector/eigenvalue, UMAP/t-SNE coordinates, plots).

Note: the PCA preparation step (Step 1) also generates its own 2D scatter plot (PC1 vs PC2) as a side effect, but it isn't tracked in `results_single['files']` — it's saved separately, as `pca_2d_plot.pdf`, directly under `pipeline.results_dir` (not the `plots/` subfolder used by UMAP/t-SNE, and always PDF regardless of `plot_format`).

In [32]:
results_single['files']

{'eigenvector': '/home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/dimensionality_reduction_results/1KG_GRCh38_variant_qc-pca.eigenvec',
 'eigenvalue': '/home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/dimensionality_reduction_results/1KG_GRCh38_variant_qc-pca.eigenval',
 'umap_coordinates': '/home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/dimensionality_reduction_results/umap_results/umap_coordinates.tsv',
 'tsne_coordinates': '/home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/dimensionality_reduction_results/tsne_results/tsne_coordinates.tsv',
 'plots': {'umap': '/home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/dimensionality_reduction_results/plots/umap_plot.svg',
  'tsne': '/home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/dimensionality_reduction_results/plots/tsne_plot.svg'}}

In [33]:
pca_plot_path = pipeline.results_dir / 'pca_2d_plot.pdf'
print(f"PCA scatter plot (PC1 vs PC2): {pca_plot_path} (exists: {pca_plot_path.exists()})")

PCA scatter plot (PC1 vs PC2): /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/dimensionality_reduction_results/pca_2d_plot.pdf (exists: False)


The UMAP and t-SNE coordinates are also kept in memory on the pipeline object, so they can be inspected directly without re-reading from disk.

Now let's explore the parameter space. Passing **lists** instead of scalars for `umap_params`/`tsne_params` makes `execute_dimensionality_reduction_pipeline()` automatically switch to a grid search over every combination (via `execute_parameter_grid()` internally). Since `force_pca_recompute=False` by default, the PCA computed above is reused — only the UMAP/t-SNE step is repeated for each combination.

We capture this cell's output into `dimred_grid_log` to keep the notebook readable — run `dimred_grid_log.show()` in a new cell if you need to inspect it.

In [34]:
umap_grid_params = {
    'n_neighbors': [10, 15, 20],
    'metric': ['euclidean'],
    'min_dist': [0.05, 0.1, 0.2],
    'random_state': [42],
}

tsne_grid_params = {
    'perplexity': [25.0, 35.0, 50.0],
    'random_state': [42],
}

In [35]:
log_path = "logs/dim_reduc.log"

with open(log_path, "w") as f:
    old_stdout_fd = os.dup(1)
    old_stderr_fd = os.dup(2)
    
    try:
        os.dup2(f.fileno(), 1)
        os.dup2(f.fileno(), 2)

        results_grid = pipeline.execute_dimensionality_reduction_pipeline(
            pca_params=pca_params,
            run_umap=True,
            umap_params=umap_grid_params,
            run_tsne=True,
            tsne_params=tsne_grid_params,
            color_hue_file=color_hue_file,
            case_control_markers=case_control_marker,
            plot_format='svg',
        )

    finally:
        os.dup2(old_stdout_fd, 1)
        os.dup2(old_stderr_fd, 2)
        os.close(old_stdout_fd)
        os.close(old_stderr_fd)

print(f"Pipeline complete. Log saved to: {log_path}")

INFO:ideal_genom.population.projection:================================================================================
INFO:ideal_genom.population.projection:STARTING FULL DIMENSIONALITY REDUCTION PIPELINE
INFO:ideal_genom.population.projection:================================================================================
INFO:ideal_genom.population.projection:Parameter grid search detected - running grid exploration
INFO:ideal_genom.population.projection:PCA files already exist, skipping PCA preparation
INFO:ideal_genom.population.projection:Using existing eigenvector file: /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/dimensionality_reduction_results/1KG_GRCh38_variant_qc-pca.eigenvec
INFO:ideal_genom.population.projection:Using existing eigenvalue file: /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/dimensionality_reduction_results/1KG_GRCh38_variant_qc-pca.eigenval
INFO:ideal_genom.population.projection:===========================

[t-SNE] Computing 76 nearest neighbors...
[t-SNE] Indexed 200 samples in 0.001s...
[t-SNE] Computed neighbors for 200 samples in 0.012s...
[t-SNE] Computed conditional probabilities for sample 200 / 200
[t-SNE] Mean sigma: 0.104054
[t-SNE] KL divergence after 250 iterations with early exaggeration: 62.910336


INFO:ideal_genom.population.projection:t-SNE reduction completed: 200 samples, 2 dimensions
INFO:ideal_genom.population.projection:t-SNE results saved to /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/dimensionality_reduction_results/tsne_results/tsne_coordinates.tsv
INFO:ideal_genom.population.projection:No metadata provided
INFO:ideal_genom.population.projection:No metadata to merge. Plotting 200 samples
INFO:ideal_genom.population.projection:Generating 2D plot: tsne_perplexity25.0_plot.svg
INFO:ideal_genom.population.projection:Plot dimensions: tsne_1 vs tsne_2


[t-SNE] KL divergence after 1000 iterations: 0.400354


INFO:ideal_genom.population.projection:Plot saved to /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/dimensionality_reduction_results/tsne_grid_results/tsne_perplexity25.0_plot.svg
INFO:ideal_genom.population.projection:t-SNE 2/3: tsne_perplexity35.0
INFO:ideal_genom.population.projection:================================================================================
INFO:ideal_genom.population.projection:STEP 3: t-SNE Reduction
INFO:ideal_genom.population.projection:================================================================================
INFO:ideal_genom.population.projection:Performing t-SNE reduction
INFO:ideal_genom.population.projection:Parameters: n_components=2, perplexity=35.0, learning_rate=200.0
INFO:ideal_genom.population.projection:Eigenvector file loaded from /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/dimensionality_reduction_results/1KG_GRCh38_variant_qc-pca.eigenvec
INFO:ideal_genom.population.projection:Eigenve

...

[t-SNE] Computing 151 nearest neighbors...
[t-SNE] Indexed 200 samples in 0.001s...
[t-SNE] Computed neighbors for 200 samples in 0.017s...
[t-SNE] Computed conditional probabilities for sample 200 / 200
[t-SNE] Mean sigma: 0.135154
[t-SNE] KL divergence after 250 iterations with early exaggeration: 56.406960


INFO:ideal_genom.population.projection:t-SNE reduction completed: 200 samples, 2 dimensions
INFO:ideal_genom.population.projection:t-SNE results saved to /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/dimensionality_reduction_results/tsne_results/tsne_coordinates.tsv
INFO:ideal_genom.population.projection:No metadata provided
INFO:ideal_genom.population.projection:No metadata to merge. Plotting 200 samples
INFO:ideal_genom.population.projection:Generating 2D plot: tsne_perplexity50.0_plot.svg
INFO:ideal_genom.population.projection:Plot dimensions: tsne_1 vs tsne_2


[t-SNE] KL divergence after 1000 iterations: 0.255417


INFO:ideal_genom.population.projection:Plot saved to /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/dimensionality_reduction_results/tsne_grid_results/tsne_perplexity50.0_plot.svg
INFO:ideal_genom.population.projection:UMAP grid summary saved to: /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/dimensionality_reduction_results/umap_grid_summary.tsv
INFO:ideal_genom.population.projection:t-SNE grid summary saved to: /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/dimensionality_reduction_results/tsne_grid_summary.tsv
INFO:ideal_genom.population.projection:================================================================================
INFO:ideal_genom.population.projection:PARAMETER GRID EXPLORATION COMPLETED
INFO:ideal_genom.population.projection:================================================================================
INFO:ideal_genom.population.projection:Total combinations explored: 12
INFO:ideal_genom.popul

Pipeline complete. Log saved to: logs/dim_reduc.log


In [36]:
print(f"Parameter grid search completed. Summaries written to: {pipeline.results_dir / 'umap_grid_summary.tsv'} and {pipeline.results_dir / 'tsne_grid_summary.tsv'}")

Parameter grid search completed. Summaries written to: /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/dimensionality_reduction_results/umap_grid_summary.tsv and /home/luis/CGE/ideal-genom-qc/ideal_genom/data/example_data/outputData/dimensionality_reduction_results/tsne_grid_summary.tsv


The grid search writes separate `umap_grid_summary.tsv` and `tsne_grid_summary.tsv` files — one per method, with only that method's columns — so no NaN placeholders appear for parameters that don't apply. Load them to compare runs.

In [37]:
umap_summary = pd.read_csv(pipeline.results_dir / 'umap_grid_summary.tsv', sep='\t')
print("UMAP grid summary:")
display(umap_summary)

tsne_summary = pd.read_csv(pipeline.results_dir / 'tsne_grid_summary.tsv', sep='\t')
print("t-SNE grid summary:")
display(tsne_summary)

UMAP grid summary:


,method,n_neighbors,metric,min_dist,random_state,config_name,n_samples
0,UMAP,10,euclidean,0.05,42,umap_n_neighbors10_metriceuclidean_min_dist0.05,200
1,UMAP,10,euclidean,0.10,42,umap_n_neighbors10_metriceuclidean_min_dist0.1,200
2,UMAP,10,euclidean,0.20,42,umap_n_neighbors10_metriceuclidean_min_dist0.2,200
3,UMAP,15,euclidean,0.05,42,umap_n_neighbors15_metriceuclidean_min_dist0.05,200
4,UMAP,15,euclidean,0.10,42,umap_n_neighbors15_metriceuclidean_min_dist0.1,200
5,UMAP,15,euclidean,0.20,42,umap_n_neighbors15_metriceuclidean_min_dist0.2,200
6,UMAP,20,euclidean,0.05,42,umap_n_neighbors20_metriceuclidean_min_dist0.05,200
7,UMAP,20,euclidean,0.10,42,umap_n_neighbors20_metriceuclidean_min_dist0.1,200
8,UMAP,20,euclidean,0.20,42,umap_n_neighbors20_metriceuclidean_min_dist0.2,200


t-SNE grid summary:


,method,perplexity,random_state,config_name,n_samples
0,t-SNE,25.0,42,tsne_perplexity25.0,200
1,t-SNE,35.0,42,tsne_perplexity35.0,200
2,t-SNE,50.0,42,tsne_perplexity50.0,200


Coordinates and plots for every combination are saved under `umap_grid_results/` and `tsne_grid_results/` inside the pipeline's results directory, named after their parameters (e.g. `umap_n_neighbors15_metriceuclidean_min_dist0.1_plot.svg`). Let's list what was generated for UMAP.

In [38]:
sorted(p.name for p in (pipeline.results_dir / 'umap_grid_results').glob('*_plot.svg'))

['umap_n_neighbors10_metricchebyshev_min_dist0.01_plot.svg',
 'umap_n_neighbors10_metricchebyshev_min_dist0.1_plot.svg',
 'umap_n_neighbors10_metricchebyshev_min_dist0.2_plot.svg',
 'umap_n_neighbors10_metriceuclidean_min_dist0.01_plot.svg',
 'umap_n_neighbors10_metriceuclidean_min_dist0.05_plot.svg',
 'umap_n_neighbors10_metriceuclidean_min_dist0.1_plot.svg',
 'umap_n_neighbors10_metriceuclidean_min_dist0.2_plot.svg',
 'umap_n_neighbors15_metricchebyshev_min_dist0.01_plot.svg',
 'umap_n_neighbors15_metricchebyshev_min_dist0.1_plot.svg',
 'umap_n_neighbors15_metricchebyshev_min_dist0.2_plot.svg',
 'umap_n_neighbors15_metriceuclidean_min_dist0.01_plot.svg',
 'umap_n_neighbors15_metriceuclidean_min_dist0.05_plot.svg',
 'umap_n_neighbors15_metriceuclidean_min_dist0.1_plot.svg',
 'umap_n_neighbors15_metriceuclidean_min_dist0.2_plot.svg',
 'umap_n_neighbors20_metricchebyshev_min_dist0.01_plot.svg',
 'umap_n_neighbors20_metricchebyshev_min_dist0.1_plot.svg',
 'umap_n_neighbors20_metricchebys

Note: unlike sample/variant/ancestry QC, there is no dedicated clean-up class for this module — intermediate PLINK files from the LD pruning + PCA step remain in `pipeline.results_dir` for inspection.